# Used Car Data Preprocessing – Day 12

Complete preprocessing workflow for the provided Used Car Resale Dataset.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

## 2. Load Dataset

In [ ]:
df = pd.read_csv("Day12_Used_Car_Preprocessing_Dataset.csv")
display(df.head())
print("Dataset shape:", df.shape)

## 3. Initial Inspection

In [ ]:
print("Columns:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:", df.duplicated().sum())

print("\nSummary Statistics:")
display(df.describe(include="all").T)

## 4. Separate Features and Target

The target variable is **`Resale_Price_Lakh`**.

In [ ]:
target = "Resale_Price_Lakh"
X = df.drop(columns=[target])
y = df[target]

print("Target:", target)
print("Number of features:", X.shape[1])

## 5. Identify Feature Types

In [ ]:
numeric_features = X.select_dtypes(include="number").columns.tolist()
categorical_features = X.select_dtypes(exclude="number").columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

## 6. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

## 7. Outlier Detection using IQR

In [ ]:
outlier_results = []

for column in numeric_features:
    values = X_train[column].dropna()
    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    count = ((values < lower) | (values > upper)).sum()

    outlier_results.append({
        "Feature": column,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "Lower_Bound": lower,
        "Upper_Bound": upper,
        "Outlier_Count": int(count)
    })

outlier_df = pd.DataFrame(outlier_results)
display(outlier_df)

## 8. Build Preprocessing Pipeline

Numeric features use median imputation and StandardScaler. Categorical features use most-frequent imputation and One-Hot Encoding.

In [ ]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

## 9. Fit on Training Data Only

In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()

X_train_processed = pd.DataFrame(
    X_train_processed, columns=feature_names, index=X_train.index
)
X_test_processed = pd.DataFrame(
    X_test_processed, columns=feature_names, index=X_test.index
)

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

display(X_train_processed.head())

## 10. Add Target and Split Labels

In [ ]:
train_processed = X_train_processed.copy()
test_processed = X_test_processed.copy()

train_processed[target] = y_train
test_processed[target] = y_test

train_processed["Dataset_Split"] = "Train"
test_processed["Dataset_Split"] = "Test"

final_processed = pd.concat([train_processed, test_processed]).sort_index()

display(final_processed.head())

## 11. Verify Processed Dataset

In [ ]:
print("Final shape:", final_processed.shape)
print("Total missing values:", final_processed.isnull().sum().sum())
print("Duplicate rows:", final_processed.duplicated().sum())

print("\nDataset split:")
print(final_processed["Dataset_Split"].value_counts())

print("\nProcessed feature sample:")
display(final_processed.head())

## 12. Export Final Dataset

In [ ]:
final_processed.to_csv(
    "preprocessed_used_car_dataset.csv",
    index=False
)

print("Preprocessed dataset saved successfully!")

## 13. Preprocessing Decisions

- **Outliers:** IQR was used to identify potential outliers in numeric features. Extreme used-car values were not automatically deleted because they may represent legitimate vehicles.
- **Missing numeric values:** Median imputation was used because it is less affected by extreme values.
- **Missing categorical values:** Most-frequent imputation was used.
- **Categorical encoding:** One-Hot Encoding was used for nominal categorical variables.
- **Feature scaling:** StandardScaler was applied to numeric input features.
- **Data leakage prevention:** The preprocessing pipeline was fitted only on training data and then applied to the test data.

## Conclusion

The Used Car dataset was inspected, outliers were analyzed using IQR, missing values were handled, categorical features were encoded, numerical features were standardized, and the data was split into training and testing sets without data leakage. The final processed dataset was exported as a CSV file.